# Notebook 3 — Isolation Forest Anomaly Detection
Tunes: contamination · max_samples · n_estimators

Outputs: anomaly score · binary label · saved model

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import classification_report, roc_auc_score

from modules.anomaly_detector import (
    train_isolation_forest, batch_scores, batch_labels
)
from modules.preprocessing import FEATURE_COLS
from config import DATA_PATH, ISO_FOREST_PATH

print('Ready.')

## 1. Load Scaled Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
# Re-apply feature engineering columns that may be missing after CSV round-trip
numeric_feat_cols = [c for c in FEATURE_COLS if c in df.columns]
X = df[numeric_feat_cols].values
y_true = df['anomaly_label'].values
print(f'Feature matrix: {X.shape}  Anomaly rate: {y_true.mean():.3f}')

## 2. Hyperparameter Tuning Grid

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score

param_grid = {
    'contamination': [0.03, 0.05, 0.07, 0.10],
    'n_estimators':  [100, 200, 300],
    'max_samples':   [256, 512, 'auto'],
}

best_f1, best_params = 0, {}
results = []

for cont in param_grid['contamination']:
    for n_est in param_grid['n_estimators']:
        for ms in param_grid['max_samples']:
            clf = IsolationForest(contamination=cont, n_estimators=n_est,
                                  max_samples=ms, random_state=42, n_jobs=-1)
            clf.fit(X)
            preds = (clf.predict(X) == -1).astype(int)
            f1    = f1_score(y_true, preds, zero_division=0)
            results.append({'contamination': cont, 'n_estimators': n_est,
                            'max_samples': str(ms), 'f1': f1})
            if f1 > best_f1:
                best_f1, best_params = f1, {'contamination': cont,
                                            'n_estimators': n_est, 'max_samples': ms}

print(f'Best F1: {best_f1:.4f}  Params: {best_params}')

## 3. Train Best Model & Save

In [ ]:
import joblib
from sklearn.ensemble import IsolationForest

best_clf = IsolationForest(
    contamination=best_params['contamination'],
    n_estimators=best_params['n_estimators'],
    max_samples=best_params['max_samples'],
    random_state=42, n_jobs=-1
)
best_clf.fit(X)
ISO_FOREST_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_clf, ISO_FOREST_PATH)
print(f'Model saved to: {ISO_FOREST_PATH}')

## 4. Evaluation

In [ ]:
scores = batch_scores(best_clf, X)
labels = batch_labels(best_clf, X)

print(classification_report(y_true, labels, target_names=['Normal','Anomaly']))
try:
    auc = roc_auc_score(y_true, scores)
    print(f'ROC-AUC: {auc:.4f}')
except Exception as e:
    print(f'ROC-AUC skipped: {e}')

## 5. Visualisation

In [ ]:
sample = min(500, len(scores))
colors = ['#E63946' if l == 1 else '#00B4D8' for l in labels[:sample]]

fig = go.Figure(go.Bar(x=list(range(sample)), y=scores[:sample],
                       marker_color=colors))
fig.update_layout(title='Isolation Forest Anomaly Scores (first 500 samples)',
                  xaxis_title='Sample', yaxis_title='Anomaly Score',
                  template='plotly_dark', height=350)
fig.show()

# Tuning results heatmap
res_df = pd.DataFrame(results)
pivot  = res_df.groupby(['contamination','n_estimators'])['f1'].max().unstack()
px.imshow(pivot, text_auto='.3f', title='F1 Score Grid (contamination × n_estimators)',
          template='plotly_dark', color_continuous_scale='Viridis').show()

print('Notebook 3 complete.')